# **Double Q-Learning logic**

---

In this notebook, we will introduce three method of **Double Q-learning** algorithm. We will introduce each of them, and them select between them, the best in our case.

### **Introduction:**

---

Q-learning is a **reinforcement** learning algorithm that trains an **agent** to assign values to its possible actions based on its current **state**, without requiring a model of the environment (**model-free**). It can handle problems with **stochastic transitions** and rewards without requiring adaptations.

In the case of this project, we will work with a advanced algorithm introduced by ***Hado van Hasselt*** in 2010, designed to mitigate the **overestimation bias** inherent in standard Q-Learning, the **Double Q-Learning**.
It works by using two separate Q-value estimators, each of which is used to update the other. Using these independent estimators, we can unbiased Q-value estimates of the actions selected using the opposite estimator. We can thus avoid maximization bias by disentangling our updates from biased estimates.

### **Algorithm choice:**

---

As we introduced earlier, we will use a **Double DQN** agent to manage our game bots. But we won't use the 2010 Hasselt version, since it was updated in 2015 by himself and in 2018 by ***Fujimoto et al.*** with two independent estimates of the true Q value.

It will looks like this:

<p align="center">
    <img src="assets/1_qHM7PWT-0AO7yRgifz8QKA.webp" alt="algo image" width="700">
</p>

### **Agent:**

---

Initialize the agent as a regular **DQN** and add one or two methods two simlulate the compute loss.

We have the `Agent()` class. This is the main class that orchestrates the learning process. It manages both the **local Q-network** (which is actively trained) and the **target Q-network** (which is used to compute stable Q-targets). The agent has five key methods:

1. **`__init__()`**: Initializes both networks, the optimizer, and the replay memory
2. **`step()`**: Stores experiences and triggers learning every 4 steps
3. **`get_action()`**: Selects actions using epsilon-greedy exploration
4. **`learn()`**: Computes the Q-learning loss and updates the local network
5. **`soft_update()`**: Gradually updates the target network parameters


This **agent** uses two **neural networks** (local and target) to learn optimal actions
in the **LunarLander** environment using the **Deep Q-Network** (DQN) algorithm.

`step()` stores experience and trigger learning every 4 steps. It takes the current state, the taken action, a received reward, a next_state and a data that tell whether episode is finished.

In [ ]:
def step(self, state, action, reward, next_state, done):
     self.memory.push((state, action, reward, next_state, done))
     self.t_step = (self.t_step + 1) % 4
     if self.t_step == 0 and len(self.memory.memory) > 150:
         experiences = self.memory.sample(150)
         self.learn(experiences, gamma=0.99)

`get_action()` selects an action using epsilon-greedy exploration. It takes a current state and exploration probability. It returns selected action (0, 1, 2, or 3 for LunarLander).

In [ ]:
def get_action(self, state, epsilon):
    state = torch.from_numpy(state).float().unsqueeze(0)
    self.local_qnetwork.eval()
    with torch.no_grad():
        action_values = self.local_qnetwork(state)
    self.local_qnetwork.train()
    if random.random() > epsilon:
        return np.argmax(action_values.cpu().data.numpy())
    else:
        return random.choice(np.arange(self.action_size))

(**Our Learn should Like this, it's a beta version we should update it to make it linked to our project**)

In [ ]:
def learn(self, experiences):
    states, actions, rewards, next_states, dones = experiences
    states = torch.FloatTensor(states).to(self.device)
    actions = torch.LongTensor(actions).to(self.device)
    rewards = torch.FloatTensor(rewards).to(self.device)
    next_states = torch.FloatTensor(next_states).to(self.device)
    dones = torch.FloatTensor(dones)

    actions = actions.view(actions.size(0))
    dones = dones.view(dones.dize(0))

    curr_q = self.model.forward(states).gather(1, actions.view(actions.size(0), 1))
    next_q = self.target_model.forward(next_states)
    max_next_q = torch.max(next_q, 1)[0]
    max_next_q = max_next_q.view(max_next_q.size(0), 1)
    expected_q = rewards + (1 - dones) * self.gamma * max_next_q

    loss = F.mse_loss(curr_q, expected_q.detach())

    return loss